<a href="https://colab.research.google.com/github/9sushant/Project1/blob/main/run_finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mistral QLoRA Fine-tuning in Google Colab

This notebook provides the environment to run the fine-tuning script (`finetune.py`) from your GitHub repository.

**Pre-requisites:**
1. Go to **Runtime -> Change runtime type** and select **T4 GPU** or better.
2. Ensure you have a Hugging Face token and have accepted the terms for the Mistral model on the Hugging Face hub.

In [1]:
# Step 1: Clone the repository and navigate into the folder
!git clone https://github.com/9sushant/Project1.git
%cd Project1

Cloning into 'Project1'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 72 (delta 38), reused 47 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 117.19 KiB | 1.56 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/Project1


In [2]:
# Step 2: Install dependencies
!pip install -r requirements.txt
!pip install -U bitsandbytes transformers peft accelerate trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting 

In [3]:
# Step 3: Login to Hugging Face (to download the Mistral base model)
# You will need to click the link that appears and paste your access token.
from huggingface_hub import notebook_login
notebook_login()

In [4]:
!git pull


Already up to date.


In [5]:
# Step 4: Run the fine-tuning script!
!python finetune.py

Loading Model: mistralai/Mistral-7B-Instruct-v0.2
Generating train split: 577 examples [00:00, 234552.57 examples/s]
Map: 100% 577/577 [00:00<00:00, 110588.26 examples/s]
config.json: 100% 596/596 [00:00<00:00, 2.99MB/s]
tokenizer_config.json: 2.10kB [00:00, 7.27MB/s]
tokenizer.json: 1.80MB [00:00, 62.4MB/s]
tokenizer.model: 100% 493k/493k [00:00<00:00, 789kB/s] 
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.56MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 25.1kB [00:00, 56.7MB/s]
Fetching 3 files: 100% 3/3 [02:14<00:00, 44.78s/it]
Download complete: 100% 14.5G/14.5G [02:14<00:00, 108MB/s]
Loading weights: 100% 291/291 [01:00<00:00,  4.79it/s]
generation_config.json: 100% 111/111 [00:00<00:00, 502kB/s]
trainable params: 83,886,080 || all params: 7,325,618,176 || trainable%: 1.1451
Adding EOS to train dataset: 100% 577/577 [00:00<00:00, 16971.70 examples/s]
Tokenizing train dataset: 100% 577/577 [00:00<00:00, 2635.01 examples/s]
Truncating 

In [6]:
!git pull

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_path = "./mistral-finetuned"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
print("✅ Fine-tuned model loaded!")


Already up to date.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Fine-tuned model loaded!


In [7]:
def chat(question, max_new_tokens=256):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

print("✅ Chat function ready!")


✅ Chat function ready!


In [9]:
test_questions = [
    "Buddhi, Bhojan aur Kaal (Mritayu) ki utpatti ka kram kya hai?",
    "Bhojan aur neend ka aapas mein kya sambandh bataya gaya hai?",
    "Gorakhnath Bhajan Mala ke antim raag (Raag Todi 41) mein 'Sahaj Samadhi' ki avastha ko kis prakar varnit kiya gaya hai?",
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print(f"{'='*60}")
    print(f"A: {chat(q)}")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Q1: Buddhi, Bhojan aur Kaal (Mritayu) ki utpatti ka kram kya hai?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


A: Manasa (mind) se buddhi paida hoti hai, buddhi se bhojan ki khoj hoti hai, bhojan se neend (sleep) aati hai aur neend ki alasya se Kaal (Death) paida hota hai.

Q2: Bhojan aur neend ka aapas mein kya sambandh bataya gaya hai?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


A: Char tarah ke bhojan (chabane, nikalne, chatne aur chusne wale) se sharir mein nindra aur swapna ki avasthayein paida hoti hain, jo ant mein jeev ko kaal ki taraf le jaati hain.

Q3: Gorakhnath Bhajan Mala ke antim raag (Raag Todi 41) mein 'Sahaj Samadhi' ki avastha ko kis prakar varnit kiya gaya hai?
A: Sahaj Samadhi vah avastha hai jahan yogi ko kisi kanth aur chehra nahi rekha karke santosh hi rakhne lagta. Usme yogi ko sone ka zara urja feela jaane lagti hai, lekin us purab-pashchim ke deh se dher moksha prapt hota hai.


Once training is complete, the adapters are saved in `./mistral-finetuned`.
You can compress this folder and download it, or push it to Hugging Face Hub!